In [1]:
from typing import NotRequired, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

load_dotenv()
model = ChatOpenAI(model="gpt-6-luna")

In [2]:
## creating state first
class mystate(TypedDict):
    question: str
    answer: str
    systemprompt: str

In [3]:
def llm(state: mystate) -> mystate:
    question = state['question']
    systemprompt = state['systemprompt']

    with open("knowledge.txt", "r") as file:
        developer_info = file.read()

    systemprompt = systemprompt + "\n\nDeveloper information:\n" + developer_info

    response = model.invoke([
        SystemMessage(content=systemprompt),
        HumanMessage(content=question)
    ])

    state['answer'] = response.content
    return state

In [4]:
## creating graph
mygraph = StateGraph(mystate)

## creating node
mygraph.add_node('llm', llm)

mygraph.add_edge(START, 'llm')
mygraph.add_edge('llm', END)

f = mygraph.compile()

In [5]:
initialstate = {
    'question': 'what is medicine search',
    'answer': '',
    'systemprompt': 'While answering, do not guess. Use the provided information file.. If asked anything else, reply only: Ask regarding the medicine only.'
}

final = f.invoke(initialstate)
print(final['answer'])

Medicine search is the process of finding reliable information about a medicine—such as its purpose, dosage, side effects, contraindications, and interactions—using trusted medical sources and official guidelines.
